# Cognix Phase 3: Region-Specific Pooling

**Purpose:** Test whether pooling brain vertices by region recovers cognitive dimensions that whole-brain mean pooling misses — especially emotional arousal (Brain−LLaMA = −0.070 in Round 2).

**No GPU needed.** Uses cached pooled vectors (20,484-d) from Google Drive.

**Key questions:**
1. Does limbic-only similarity recover emotional arousal?
2. Does prefrontal-only similarity best separate cognitive load?
3. Does motor-only similarity best capture sensorimotor?
4. Which regions diverge most from LLaMA?
5. Should the distilled model output a structured embedding (per-region scores) or a black box vector?

In [ ]:
!pip install -q nilearn nibabel
!git clone https://github.com/gdavid7/cognix.git /content/cognix 2>/dev/null; echo 'repo ready'

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from pathlib import Path
from collections import defaultdict

PAIRS_PATH = '/content/cognix/data/validation_pairs_r2.jsonl'
POOLED_DIR = Path('/content/drive/MyDrive/cognix_cache/pooled_vectors')
LLAMA_DIR = Path('/content/drive/MyDrive/cognix_cache/llama_vectors')

def text_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

## 1. Load Destrieux atlas for fsaverage5

Map each of the 20,484 cortical vertices to a brain region.

In [ ]:
from nilearn import datasets, surface
import nibabel as nib

# Fetch Destrieux atlas for fsaverage5 (74 regions per hemisphere)
atlas = datasets.fetch_atlas_surf_destrieux(mesh='fsaverage5')

# Load label arrays
lh_labels, lh_ctab, lh_names = nib.freesurfer.read_annot(atlas['map_left'])
rh_labels, rh_ctab, rh_names = nib.freesurfer.read_annot(atlas['map_right'])

# Decode names
lh_names = [n.decode('utf-8') if isinstance(n, bytes) else n for n in lh_names]
rh_names = [n.decode('utf-8') if isinstance(n, bytes) else n for n in rh_names]

# Build combined label array (20484,) and name mapping
# Offset right hemisphere labels to avoid collisions
rh_offset = max(lh_labels) + 1
combined_labels = np.concatenate([lh_labels, rh_labels + rh_offset])

label_to_name = {}
for i, name in enumerate(lh_names):
    label_to_name[i] = f'lh_{name}'
for i, name in enumerate(rh_names):
    label_to_name[i + rh_offset] = f'rh_{name}'

print(f'Label array shape: {combined_labels.shape}')
print(f'Unique labels: {len(np.unique(combined_labels))}')
print(f'Left hemi regions: {len(lh_names)}')
print(f'Right hemi regions: {len(rh_names)}')
print(f'\nAll region names:')
for name in sorted(set(lh_names)):
    count = np.sum(np.array(lh_names)[lh_labels] == name) if name in lh_names else 0
    print(f'  {name}')

## 2. Define cognitive region groups

Group Destrieux regions into 6 cognitive categories based on neuroscience literature.

In [ ]:
# Cognitive groupings — map Destrieux region substrings to cognitive categories.
# Each vertex gets assigned to exactly one group (first match wins).
# Regions not matching any group go into 'other'.

COGNITIVE_GROUPS = {
    'prefrontal': [
        'G_front_sup', 'G_front_middle', 'G_front_inf',
        'G_orbital', 'G_rectus', 'G_and_S_transv_frontopol',
        'G_and_S_frontomargin', 'G_subcallosal',
        'S_front_sup', 'S_front_middle', 'S_front_inf',
        'S_orbital',
    ],
    'limbic': [
        'G_and_S_cingul', 'G_cingul',
        'S_cingul', 'S_pericallosal',
        'G_oc-temp_med-Parahip',
        'G_Ins_lg_and_S_cent_ins', 'G_insular_short',
        'S_circular_insula',
        'S_suborbital',
    ],
    'motor': [
        'G_precentral', 'G_postcentral',
        'G_and_S_paracentral', 'G_and_S_subcentral',
        'S_central', 'S_precentral', 'S_postcentral',
    ],
    'parietal': [
        'G_parietal_sup', 'G_pariet_inf',
        'G_precuneus',
        'S_intrapariet', 'S_subparietal',
        'G_pariet_inf-Supramar', 'G_pariet_inf-Angular',
        'S_parieto_occipital',
    ],
    'temporal': [
        'G_temp_sup', 'G_temporal_middle', 'G_temporal_inf',
        'S_temporal', 'Pole_temporal',
        'Lat_Fis',
        'S_collat_transv_ant',
    ],
    'visual': [
        'G_cuneus', 'G_occipital', 'Pole_occipital',
        'G_oc-temp_lat', 'G_oc-temp_med-Lingual',
        'S_calcarine', 'S_oc_middle', 'S_oc_sup',
        'S_oc-temp', 'S_collat_transv_post',
    ],
}

def assign_group(region_name):
    """Assign a Destrieux region to a cognitive group."""
    # Strip hemisphere prefix
    bare = region_name.split('_', 1)[1] if region_name.startswith(('lh_', 'rh_')) else region_name
    for group, patterns in COGNITIVE_GROUPS.items():
        for pattern in patterns:
            if bare.startswith(pattern) or pattern in bare:
                return group
    return 'other'

# Build vertex -> group mapping
vertex_groups = np.array([assign_group(label_to_name.get(l, 'unknown')) for l in combined_labels])

# Build group -> vertex indices
group_indices = {}
for group in list(COGNITIVE_GROUPS.keys()) + ['other']:
    group_indices[group] = np.where(vertex_groups == group)[0]

print('COGNITIVE REGION GROUPS')
print(f'{"Group":<15} {"Vertices":>8}  {"% of total":>10}')
print('-' * 40)
for group in list(COGNITIVE_GROUPS.keys()) + ['other']:
    n = len(group_indices[group])
    print(f'{group:<15} {n:>8}  {100*n/20484:>9.1f}%')
print(f'{"TOTAL":<15} {sum(len(v) for v in group_indices.values()):>8}')

## 3. Load data

In [ ]:
# Load pairs
pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))
print(f'Loaded {len(pairs)} pairs')

# Load brain vectors
brain_cache = {}
for npy_file in POOLED_DIR.glob('*.npy'):
    brain_cache[npy_file.stem] = np.load(npy_file)
print(f'Loaded {len(brain_cache)} brain vectors')

# Load LLaMA vectors
llama_cache = {}
for npy_file in LLAMA_DIR.glob('*.npy'):
    llama_cache[npy_file.stem] = np.load(npy_file)
print(f'Loaded {len(llama_cache)} LLaMA vectors')

## 4. Compute per-region similarity for all pairs

For each pair and each cognitive region group, extract the vertices belonging to that region and compute cosine similarity on the sub-vector.

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist
from sentence_transformers import SentenceTransformer

# Compute semantic similarity
st_model = SentenceTransformer('all-MiniLM-L6-v2')
semantic_sims = np.array([
    1.0 - cosine_dist(st_model.encode(p['text_a']), st_model.encode(p['text_b']))
    for p in pairs
])

# Compute whole-brain, LLaMA, and per-region similarities
region_names = list(COGNITIVE_GROUPS.keys())
whole_brain_sims = []
llama_sims = []
region_sims = {r: [] for r in region_names}

for p in pairs:
    ha = text_hash(p['text_a'])
    hb = text_hash(p['text_b'])
    va = brain_cache.get(ha)
    vb = brain_cache.get(hb)
    la = llama_cache.get(ha)
    lb = llama_cache.get(hb)

    # Whole brain
    if va is not None and vb is not None:
        whole_brain_sims.append(1.0 - cosine_dist(va, vb))
    else:
        whole_brain_sims.append(np.nan)

    # LLaMA
    if la is not None and lb is not None:
        llama_sims.append(1.0 - cosine_dist(la, lb))
    else:
        llama_sims.append(np.nan)

    # Per-region
    for region in region_names:
        idx = group_indices[region]
        if va is not None and vb is not None and len(idx) > 0:
            region_sims[region].append(1.0 - cosine_dist(va[idx], vb[idx]))
        else:
            region_sims[region].append(np.nan)

whole_brain_sims = np.array(whole_brain_sims)
llama_sims = np.array(llama_sims)
for r in region_names:
    region_sims[r] = np.array(region_sims[r])

print(f'Computed similarities for {len(pairs)} pairs across {len(region_names)} regions')
print(f'\nMean similarity per region (all pairs):')
for r in region_names:
    print(f'  {r:<15} {np.nanmean(region_sims[r]):.3f} (std={np.nanstd(region_sims[r]):.3f})')
print(f'  {"whole_brain":<15} {np.nanmean(whole_brain_sims):.3f}')
print(f'  {"llama":<15} {np.nanmean(llama_sims):.3f}')
print(f'  {"semantic":<15} {np.nanmean(semantic_sims):.3f}')

## 5. The key test: which region best captures each cognitive category?

For each handcrafted divergence category, compute the mean similarity per region. If region pooling works, we should see:
- Limbic highest for emotional arousal
- Prefrontal highest for cognitive load
- Motor highest for sensorimotor
- Parietal/visual highest for spatial scene

In [ ]:
divergence_cats = ['cognitive_load', 'emotional_arousal', 'sensorimotor',
                   'spatial_scene', 'syntactic_complexity', 'narrative_suspense',
                   'theory_of_mind']

# For each category, show mean similarity per region
print('MEAN SIMILARITY PER REGION PER CATEGORY')
header = f'{"Category":<25}' + ''.join(f'{r:>12}' for r in region_names) + f'{"whole":>12}{"llama":>12}{"semantic":>12}'
print(header)
print('-' * len(header))

for cat in divergence_cats + ['paraphrase', 'unrelated', 'control_same_topic_diff_processing']:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    row = f'{cat:<25}'
    for r in region_names:
        row += f'{np.nanmean(region_sims[r][mask]):>12.3f}'
    row += f'{np.nanmean(whole_brain_sims[mask]):>12.3f}'
    row += f'{np.nanmean(llama_sims[mask]):>12.3f}'
    row += f'{np.nanmean(semantic_sims[mask]):>12.3f}'
    print(row)

In [ ]:
# For each category, which region has the HIGHEST similarity?
# And which region diverges most from LLaMA?

print('BEST REGION PER CATEGORY')
print(f'{"Category":<25} {"Best region":>15} {"Region sim":>10} {"Whole brain":>12} {"LLaMA":>8} {"Region−LLaMA":>13}')
print('-' * 90)

for cat in divergence_cats:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue

    best_region = max(region_names, key=lambda r: np.nanmean(region_sims[r][mask]))
    best_sim = np.nanmean(region_sims[best_region][mask])
    whole = np.nanmean(whole_brain_sims[mask])
    llama = np.nanmean(llama_sims[mask])
    diff = best_sim - llama

    print(f'{cat:<25} {best_region:>15} {best_sim:>10.3f} {whole:>12.3f} {llama:>8.3f} {diff:>+13.3f}')

## 6. The critical test: does limbic pooling recover emotional arousal?

Emotional arousal was the only category where whole-brain brain sim < LLaMA sim (−0.070). If limbic-only pooling shows brain sim > LLaMA sim for this category, region pooling adds real value.

In [ ]:
emo_mask = np.array([p['category'] == 'emotional_arousal' and p['source'] == 'handcrafted' for p in pairs])

print('EMOTIONAL AROUSAL — REGION COMPARISON')
print(f'  N pairs: {emo_mask.sum()}')
print(f'  Semantic sim:      {np.nanmean(semantic_sims[emo_mask]):.3f}')
print(f'  LLaMA sim:         {np.nanmean(llama_sims[emo_mask]):.3f}')
print(f'  Whole brain sim:   {np.nanmean(whole_brain_sims[emo_mask]):.3f}  (Brain−LLaMA: {np.nanmean(whole_brain_sims[emo_mask]) - np.nanmean(llama_sims[emo_mask]):+.3f})')
print()
for r in region_names:
    rsim = np.nanmean(region_sims[r][emo_mask])
    ldiff = rsim - np.nanmean(llama_sims[emo_mask])
    marker = ' <<<' if r == 'limbic' else ''
    print(f'  {r:<15} sim: {rsim:.3f}  (Region−LLaMA: {ldiff:+.3f}){marker}')

print()
limbic_emo = np.nanmean(region_sims['limbic'][emo_mask])
whole_emo = np.nanmean(whole_brain_sims[emo_mask])
llama_emo = np.nanmean(llama_sims[emo_mask])
if limbic_emo > llama_emo and whole_emo < llama_emo:
    print('RESULT: Limbic pooling RECOVERS emotional arousal signal that whole-brain pooling missed.')
elif limbic_emo > whole_emo:
    print('RESULT: Limbic pooling improves emotional arousal over whole-brain, but may not fully recover it.')
else:
    print('RESULT: Limbic pooling does NOT improve emotional arousal. The signal may not be region-specific.')

## 7. Per-region divergence from LLaMA (all categories)

For each region, compute Brain−LLaMA divergence per category. This shows which regions contribute most beyond what LLaMA already captures.

In [ ]:
print('REGION−LLaMA DIVERGENCE PER CATEGORY')
print('(Positive = region sees MORE similarity than LLaMA)')
header = f'{"Category":<25}' + ''.join(f'{r:>12}' for r in region_names) + f'{"whole":>12}'
print(header)
print('-' * len(header))

for cat in divergence_cats + ['paraphrase', 'unrelated']:
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    llama_mean = np.nanmean(llama_sims[mask])
    row = f'{cat:<25}'
    for r in region_names:
        diff = np.nanmean(region_sims[r][mask]) - llama_mean
        row += f'{diff:>+12.3f}'
    whole_diff = np.nanmean(whole_brain_sims[mask]) - llama_mean
    row += f'{whole_diff:>+12.3f}'
    print(row)

## 8. Region correlation with semantic similarity

Which region diverges most from semantic similarity? Low correlation = captures something different.

In [ ]:
print('PEARSON r: REGION SIMILARITY vs SEMANTIC SIMILARITY')
print(f'{"Region":<15} {"Pearson r":>10} {"p-value":>12}')
print('-' * 40)

valid = ~np.isnan(whole_brain_sims)
for r in region_names:
    rvalid = valid & ~np.isnan(region_sims[r])
    if rvalid.sum() > 10:
        rp, pp = stats.pearsonr(semantic_sims[rvalid], region_sims[r][rvalid])
        print(f'{r:<15} {rp:>10.4f} {pp:>12.2e}')

rp, pp = stats.pearsonr(semantic_sims[valid], whole_brain_sims[valid])
print(f'{"whole_brain":<15} {rp:>10.4f} {pp:>12.2e}')
rp, pp = stats.pearsonr(semantic_sims[valid], llama_sims[valid])
print(f'{"llama":<15} {rp:>10.4f} {pp:>12.2e}')

## 9. Heatmap: region similarity by category

In [ ]:
cats_to_plot = divergence_cats + ['control_same_topic_diff_processing', 'paraphrase', 'unrelated']
regions_to_plot = region_names + ['whole_brain']

# Build matrix
matrix = np.zeros((len(cats_to_plot), len(regions_to_plot)))
for i, cat in enumerate(cats_to_plot):
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    llama_mean = np.nanmean(llama_sims[mask])
    for j, r in enumerate(regions_to_plot):
        if r == 'whole_brain':
            matrix[i, j] = np.nanmean(whole_brain_sims[mask]) - llama_mean
        else:
            matrix[i, j] = np.nanmean(region_sims[r][mask]) - llama_mean

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(matrix, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(regions_to_plot)))
ax.set_xticklabels(regions_to_plot, rotation=45, ha='right')
ax.set_yticks(range(len(cats_to_plot)))
ax.set_yticklabels(cats_to_plot)
ax.set_title('Region−LLaMA divergence per category\n(Blue = brain < LLaMA, Red = brain > LLaMA)')
plt.colorbar(im, label='Brain region sim − LLaMA sim')

# Add values
for i in range(len(cats_to_plot)):
    for j in range(len(regions_to_plot)):
        ax.text(j, i, f'{matrix[i,j]:+.3f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cognix_cache/region_heatmap.png', dpi=150)
plt.show()

## 10. Summary and decision

In [ ]:
print('PHASE 3 SUMMARY')
print('=' * 60)

# Check if limbic recovered emotional arousal
emo_mask = np.array([p['category'] == 'emotional_arousal' and p['source'] == 'handcrafted' for p in pairs])
limbic_recovered = np.nanmean(region_sims['limbic'][emo_mask]) > np.nanmean(llama_sims[emo_mask])

# Check if regions show category-specific patterns
expected_best = {
    'cognitive_load': 'prefrontal',
    'emotional_arousal': 'limbic',
    'sensorimotor': 'motor',
    'spatial_scene': 'parietal',
}
matches = 0
for cat, expected_region in expected_best.items():
    mask = np.array([p['category'] == cat and p['source'] == 'handcrafted' for p in pairs])
    if mask.sum() < 3:
        continue
    actual_best = max(region_names, key=lambda r: np.nanmean(region_sims[r][mask]))
    match = actual_best == expected_region
    matches += int(match)
    status = 'MATCH' if match else f'MISMATCH (got {actual_best})'
    print(f'  {cat}: expected {expected_region}, {status}')

print(f'\n  Region-category matches: {matches}/{len(expected_best)}')
print(f'  Limbic recovered emotional arousal: {limbic_recovered}')

print()
if matches >= 3 and limbic_recovered:
    print('DECISION: Region pooling shows clear category-specific patterns.')
    print('  -> Distilled model should output STRUCTURED embedding with per-region scores.')
elif matches >= 2 or limbic_recovered:
    print('DECISION: Region pooling shows partial patterns.')
    print('  -> Consider structured embedding but include whole-brain as a fallback dimension.')
else:
    print('DECISION: Region pooling does not show clear patterns.')
    print('  -> Distilled model should output BLACK BOX embedding (single vector).')
    print('  -> The brain signal is diffuse, not region-specific.')

In [ ]:
# Save region similarity results
results = []
for i, p in enumerate(pairs):
    entry = {
        'id': p['id'],
        'source': p['source'],
        'category': p['category'],
        'sim_semantic': float(semantic_sims[i]),
        'sim_llama': float(llama_sims[i]) if not np.isnan(llama_sims[i]) else None,
        'sim_whole_brain': float(whole_brain_sims[i]) if not np.isnan(whole_brain_sims[i]) else None,
    }
    for r in region_names:
        entry[f'sim_{r}'] = float(region_sims[r][i]) if not np.isnan(region_sims[r][i]) else None
    results.append(entry)

with open('/content/drive/MyDrive/cognix_cache/r3_region_results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f'Saved {len(results)} results to Drive')